# R6 — Untouched confirmation test on NF-ToN-IoT-v2 (asymmetric abstention + BCDR). POST HOC, pre-registered.
Uses the models, calibration and thresholds saved by R5 **without retraining**. Run G0 → G1 → G2 (GPU runtime recommended).

### CELL G0 — save and verify the frozen remedy protocol, load frozen R5 artifacts

In [ ]:
# CELL G0 — save and verify the frozen remedy protocol, load frozen R5 artifacts
from google.colab import drive
drive.mount("/content/drive")
import os, json, hashlib, gc, shutil, time
import numpy as np, pandas as pd, xgboost as xgb, keras
BASE = "/content/drive/MyDrive/TrustShift_IDS"
R5 = f"{BASE}/V2_adaptive_reliability/revision_R5_netflow"
OUT = f"{BASE}/V2_adaptive_reliability/revision_R6_confirmation"; os.makedirs(OUT, exist_ok=True)
TON = f"{BASE}/data/raw/NF-ToN-IoT-v2/NF-ToN-IoT-v2.csv"
PROTO = '{\n  "created": "2026-10-22 12:26:07 UTC",\n  "status": "Written before computing any asymmetric-policy or metric result on any dataset.",\n  "metric": {\n    "name": "Budget-compliant detection rate (BCDR)",\n    "definition": "Share of all attack flows that are correctly detected (accepted and predicted attack) in windows whose selective risk is at or below the 1% budget. Detections in violating windows count as zero, because their decisions cannot be trusted; deferred attacks count as zero, because they receive no automatic decision.",\n    "reported_with": [\n      "violating windows",\n      "overall coverage",\n      "attack deferral",\n      "false-alarm rate among accepted benign flows"\n    ]\n  },\n  "policies": {\n    "SYM(t)": "Existing symmetric policy: accept any decision with risk score <= t.",\n    "ASY(t)": "Asymmetric policy: always accept \'attack\' verdicts (they become analyst-reviewed alerts); accept \'benign\' verdicts only if risk score <= t; defer the remaining benign verdicts.",\n    "thresholds": "t in {Elevated, Severe} state thresholds of each detector, set from calibration data only; plus the monitor-adaptive variant in which the state (TrustShift states for CICIDS2017 XGBoost and neural detectors, ATC for NetFlow) chooses t per window."\n  },\n  "datasets": {\n    "existing": [\n      "CICIDS2017 deployment sequences and CSE-CIC-IDS2018 (XGBoost, MLP, FT-Transformer)",\n      "NF-UNSW-NB15-v2 sequences and NF-CSE-CIC-IDS2018-v2 (XGBoost, MLP)"\n    ],\n    "confirmation": "NF-ToN-IoT-v2 as an untouched external test for the NF-UNSW-NB15-v2 detectors; evaluated only after this protocol and the existing-data results are fixed."\n  },\n  "decision_rule": "The remedy is judged useful if, on the untouched confirmation test, ASY raises BCDR over the best symmetric policy for both detectors without increasing violating windows. Any other outcome is reported as a negative result.",\n  "computation": "For each window, ASY counts are derived from the fixed (accept-all) record for attack verdicts and from the threshold-t record for benign verdicts."\n}'
assert hashlib.sha256(PROTO.encode()).hexdigest() == "488d563fc1e83fdc7709dbb47abd507f2991080bb272f307c05c0bc6cfba80ca", "protocol text changed"
open(f"{OUT}/REMEDY_PROTOCOL_FROZEN.json", "w").write(PROTO)
json.dump({"saved": time.strftime("%Y-%m-%d %H:%M:%S"), "note": "protocol saved and verified before reading NF-ToN-IoT-v2"}, open(f"{OUT}/G0_protocol_check.json", "w"))
print("[G0] remedy protocol verified (sha256 488d563f…) and saved before any confirmation data is read")

R5P = json.load(open(f"{R5}/F0_PROTOCOL_FROZEN.json")); FEATURES = R5P["features"]; W = R5P["window"]; EPS = R5P["risk_budget"]
CAL = json.load(open(f"{R5}/F1_calibration_thresholds.json"))
xgb_m = xgb.XGBClassifier(); xgb_m.load_model(f"{R5}/F1_xgb_source.json")
mlp = keras.models.load_model(f"{R5}/F1_mlp_source.keras", compile=False)
prep = np.load(f"{R5}/F1_mlp_prep.npz"); mu, sd = prep["mu"], prep["sd"]
cols = pd.read_csv(TON, nrows=2).columns.tolist()
assert all(f in cols for f in FEATURES) and "Label" in cols, "NF-ToN-IoT-v2 schema differs"
print("[G0] frozen R5 models, calibration and thresholds loaded; schema matches (38 features)")

F32MAX = np.finfo(np.float32).max; SAT = {"values": 0, "rows": 0}
def p_attack(det, X, bs=65536):
    if det == "XGBoost":
        bad = ~np.isfinite(X) | (np.abs(X) > F32MAX)
        if bad.any():
            SAT["values"] += int(bad.sum()); SAT["rows"] += int(bad.any(axis=1).sum())
            X = np.where(np.isnan(X), 0.0, np.clip(X, -F32MAX, F32MAX))
        return xgb_m.predict_proba(X.astype(np.float32))[:, 1].astype(np.float64)
    Z = ((np.log1p(np.clip(X, 0, None)) - mu) / sd).astype(np.float32)
    return mlp.predict(Z, batch_size=bs, verbose=0).reshape(-1).astype(np.float64)
def logit(p): p = np.clip(p, 1e-12, 1 - 1e-12); return np.log(p) - np.log1p(-p)
def risk_pred(p, T):
    q = 1 / (1 + np.exp(-logit(p) / T)); return 1 - np.maximum(q, 1 - q), (q >= 0.5).astype(np.int64)
def records(r, pred, y, base, det):
    conf = 1 - r; att = y == 1; err = pred != y; rows = []
    atc = float(np.mean(conf < CAL[det]["atc_conf_threshold"])); mr = float(r.mean())
    for tn, t in CAL[det]["thresholds"].items():
        acc = r <= t
        rows.append({**base, "detector": det, "thr_name": tn, "atc_est": atc, "mean_risk": mr, "n": len(y), "acc_n": int(acc.sum()),
                     "acc_err": int((acc & err).sum()), "n_attack": int(att.sum()), "acc_attack": int((acc & att).sum()),
                     "acc_missed_attack": int((acc & att & (pred == 0)).sum()), "acc_benign": int((acc & ~att).sum()),
                     "acc_false_alarm": int((acc & ~att & (pred == 1)).sum())})
    return rows
print("[G0] COMPLETE")

### CELL G1 — confirmation test: all NF-ToN-IoT-v2 flows in file order, 5,000-flow windows

In [ ]:
# CELL G1 — confirmation test: all NF-ToN-IoT-v2 flows in file order, 5,000-flow windows
rows = []; carry = None; wid = 0; tot = {d: {"n": 0, "err": 0, "att": 0, "det": 0, "fa": 0, "ben": 0} for d in ["XGBoost", "MLP"]}
for ch in pd.read_csv(TON, usecols=FEATURES + ["Label"], chunksize=1_000_000):
    Xc = ch[FEATURES].to_numpy(np.float64); yc = ch["Label"].to_numpy(np.int64)
    if carry is not None: Xc = np.vstack([carry[0], Xc]); yc = np.concatenate([carry[1], yc])
    nfull = (len(yc) // W) * W; carry = (Xc[nfull:], yc[nfull:]); Xc, yc = Xc[:nfull], yc[:nfull]
    for det in ["XGBoost", "MLP"]:
        r, pr = risk_pred(p_attack(det, Xc), CAL[det]["T"]); t = tot[det]
        t["n"] += len(yc); t["err"] += int((pr != yc).sum()); t["att"] += int(yc.sum()); t["det"] += int(((pr == 1) & (yc == 1)).sum())
        t["fa"] += int(((pr == 1) & (yc == 0)).sum()); t["ben"] += int((yc == 0).sum())
        for w in range(nfull // W):
            a, b = w * W, (w + 1) * W
            rows += records(r[a:b], pr[a:b], yc[a:b], {"window_id": wid + w}, det)
    wid += nfull // W; print(f"  windows so far: {wid:,}")
for det, t in tot.items():
    print(f"[G1] {det}: flows {t['n']:,} | accuracy {1 - t['err']/t['n']:.4f} | attack share {t['att']/t['n']:.4f} | "
          f"attack recall {t['det']/t['att']:.4f} | false-alarm rate {t['fa']/t['ben']:.4f}")
print("[G1] saturated values (XGBoost path):", SAT, "| tail flows not windowed:", len(carry[1]))
json.dump({"saturated": SAT, "totals": tot}, open(f"{OUT}/G1_totals.json", "w"), indent=2)
pd.DataFrame(rows).to_csv(f"{OUT}/G1_ToN_window_outcomes.csv.gz", index=False); print("[G1] COMPLETE", len(rows))

### CELL G2 — package (zip written outside the folder being zipped)

In [ ]:
# CELL G2 — package (zip written outside the folder being zipped)
for f in os.listdir(OUT):
    if f.endswith(".zip"): os.remove(os.path.join(OUT, f))
z = shutil.make_archive(f"{BASE}/V2_adaptive_reliability/TrustShift_revision_R6_export", "zip", OUT)
print("Upload:", z, f"({os.path.getsize(z)/1e6:.1f} MB)"); print("Contents:", sorted(os.listdir(OUT)))